In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"  # "20251028_140930" # "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive

# one regressor
"""--------------------------------------------"""

## init

In [ ]:
import numpy as np

d = {
    "a": np.random.randn(10),
    "b": np.random.randn(10),
}

np.array(list(d.values())).mean(axis=0)

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id, do_ohe=False, norm=True)
encoder.verify()

In [ ]:
plt.figure()
plt.plot(encoder.trial_data["block_side"])
plt.show()

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_fits()

In [ ]:
from sklearn.preprocessing import OneHotEncoder as OHE

ohe = OHE().fit(encoder.tvs)
tvs_ohe = ohe.transform(encoder.tvs)

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeCV

encoder.baseline_predict()
resids = encoder.robs - encoder.robs_predict["baseline"]

enc = RidgeCV(
    alphas=np.logspace(-5, 5, 11, base=10),
    alpha_per_target=True,
).fit(encoder.tvs, resids)
enc.intercept_

In [ ]:
from sklearn.linear_model import LinearRegression

enc_lr = LinearRegression().fit(encoder.tvs, resids)
enc_lr.coef_

In [ ]:
fig, axes = plt.subplots(ncols=2)
axes[0].imshow(encoder.tvs, aspect="auto")
axes[1].imshow(resids)

In [ ]:
enc_predict = enc.predict(encoder.tvs) + encoder.robs_predict["baseline"]

In [ ]:
plt.figure()
plt.hist(enc.alpha_, bins=np.logspace(-5, 5, 11, base=10))
plt.xscale("log")
plt.show()

In [ ]:
plt.figure()
plt.plot(encoder.robs[:, 0])
plt.plot(enc_predict[:, 0])
plt.show()

In [ ]:
encoder.view_fits(model="baseline")

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.build_dm()
encoder.plot_r2_comp()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
for k in encoder.tv_keys:
    print(k, encoder.trial_data[k].values.shape)

In [ ]:
quot = encoder.robs / encoder.robs_predict["encoder"]

In [ ]:
from sklearn.neural_network import MLPRegressor as NN

X = encoder.robs
y = encoder.robs - encoder.robs_predict["encoder"]

sg_a_estimator = NN(hidden_layer_sizes=[1]).fit(X, y)

In [ ]:
v = sg_a_estimator.coefs_[-1].flatten()

In [ ]:
W1 = sg_a_estimator.coefs_[0]
b1 = sg_a_estimator.intercepts_[0]

m = X @ W1 + b1

vm = v * m

In [ ]:
robs_predict_sg = vm + encoder.robs_predict["encoder"]

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

r = FitRenderer(
    y=encoder.robs[:, encoder.reg_idxs[reg]] * (1000 / encoder.binwidth_ms),
    yhat=robs_predict_sg[:, encoder.reg_idxs[reg]] * (1000 / encoder.binwidth_ms),
    ylabel="Firing Rate (Hz)",
    rsquared=encoder.scores["encoder"][encoder.reg_idxs[reg]],
    mode="lite",
)

NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
plt.figure()
plt.plot(y)
plt.show()

In [ ]:
from scipy.stats import pearsonr as r

corr = {
    tv: r(m.flatten(), encoder.trial_data[tv].values).statistic
    for tv in encoder.tv_keys
}
corr

In [ ]:
encoder.trial_data["response"].values.shape

In [ ]:
encoder.view_peths()

In [ ]:
encoder_mf.view_fits()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## weight correlation

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca on DMS/DLS robs

In [ ]:
from sklearn.decomposition import PCA
from utils.colors import colors_region

pca_dls = PCA().fit(encoder.robs[:, encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.robs[:, encoder.reg_idxs["DMS"]])
cum_var_dls = np.array([sum(pca_dls.explained_variance_ratio_[:n]) for n in range(100)])
cum_var_dms = np.array([sum(pca_dms.explained_variance_ratio_[:n]) for n in range(100)])

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.legend()
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("spike counts")
plt.show()

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca_dls = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DMS"]])

cum_var_dls = np.array(
    [
        sum(pca_dls.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)
cum_var_dms = np.array(
    [
        sum(pca_dms.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("encoding beta weights")
plt.legend()
plt.show()

In [ ]:
pca = PCA().fit(encoder.encoder_weights)
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()